In [1]:

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/uditjain13/heart-disease-risk-2026/heart_disease_risk_2026.csv


# Import libraries

In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('/kaggle/input/datasets/uditjain13/heart-disease-risk-2026/heart_disease_risk_2026.csv')

# General Understanding about dataset

In [4]:
df.shape
df.info()
df.describe()
df.isnull().sum()
df.nunique()
df.duplicated().sum()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   patient_id                 9000 non-null   int64  
 1   age                        9000 non-null   int64  
 2   sex                        9000 non-null   object 
 3   resting_bp_systolic        9000 non-null   int64  
 4   resting_bp_diastolic       9000 non-null   int64  
 5   cholesterol_total          9000 non-null   int64  
 6   hdl                        9000 non-null   int64  
 7   ldl                        9000 non-null   int64  
 8   triglycerides              9000 non-null   int64  
 9   fasting_blood_sugar        9000 non-null   int64  
 10  hba1c                      9000 non-null   float64
 11  bmi                        9000 non-null   float64
 12  resting_heart_rate         9000 non-null   int64  
 13  max_heart_rate_achieved    9000 non-null   int64

,patient_id,age,sex,resting_bp_systolic,resting_bp_diastolic,cholesterol_total,hdl,ldl,triglycerides,fasting_blood_sugar,...,family_history,smoker_status,alcohol_units_per_week,exercise_minutes_per_week,sleep_hours,stress_score,wearable_owner,daily_steps,diet_quality_score,has_heart_disease
0,1,44,Male,117,74,193,57,106,119,112,...,False,Never,2.9,86,5.4,19.8,True,7731,62.9,0
1,2,57,Male,139,94,185,69,110,35,114,...,False,Never,3.0,132,4.3,45.8,True,2629,74.6,1
2,3,29,Male,128,78,197,52,108,157,95,...,False,Current,3.5,128,5.1,17.7,True,9290,65.7,0
3,4,72,Male,132,86,197,59,104,143,92,...,False,Never,2.7,18,6.8,63.6,True,7373,48.5,1
4,5,62,Female,116,75,154,65,75,104,135,...,True,Former,3.3,24,8.2,58.7,False,6331,47.3,1


# Preprocessing and cleaning (if needed)

In [5]:
# drop exact duplicate rows if any
df = df.drop_duplicates().reset_index(drop=True)

# try to auto-detect the target column (common names for this kind of dataset)
possible_targets = ['target', 'risk', 'heart_disease', 'heart_disease_risk', 'disease',
                     'HeartDisease', 'HeartDiseaseRisk', 'risk_level', 'label', 'Risk']
target_col = None
for c in possible_targets:
    if c in df.columns:
        target_col = c
        break
if target_col is None:
    # fallback: assume last column is the target
    target_col = df.columns[-1]
print('Target column detected as:', target_col)

# fill missing numeric values with median, categorical with mode
for c in df.columns:
    if df[c].isnull().sum() > 0:
        if df[c].dtype == 'object':
            df[c] = df[c].fillna(df[c].mode()[0])
        else:
            df[c] = df[c].fillna(df[c].median())

# encode all remaining categorical (object) columns except the target
from sklearn.preprocessing import LabelEncoder

cat_cols = [c for c in df.select_dtypes(include='object').columns if c != target_col]
encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c].astype(str))
    encoders[c] = le

# encode the target if it's categorical (e.g. Yes/No, Low/Medium/High)
if df[target_col].dtype == 'object':
    le_target = LabelEncoder()
    df[target_col] = le_target.fit_transform(df[target_col].astype(str))
    print('Target classes:', list(le_target.classes_))

df.info()

Target column detected as: has_heart_disease
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   patient_id                 9000 non-null   int64  
 1   age                        9000 non-null   int64  
 2   sex                        9000 non-null   int64  
 3   resting_bp_systolic        9000 non-null   int64  
 4   resting_bp_diastolic       9000 non-null   int64  
 5   cholesterol_total          9000 non-null   int64  
 6   hdl                        9000 non-null   int64  
 7   ldl                        9000 non-null   int64  
 8   triglycerides              9000 non-null   int64  
 9   fasting_blood_sugar        9000 non-null   int64  
 10  hba1c                      9000 non-null   float64
 11  bmi                        9000 non-null   float64
 12  resting_heart_rate         9000 non-null   int64  
 13  max

# Feature Engineering

In [6]:
# generic engineered features using whatever numeric columns exist
num_cols = [c for c in df.select_dtypes(include=np.number).columns if c != target_col]

# 1) average of all numeric risk indicators (a simple composite score)
if len(num_cols) > 1:
    df['risk_score_mean'] = df[num_cols].mean(axis=1)

# 2) age-based ratio feature, if an age-like column exists
age_col = next((c for c in df.columns if 'age' in c.lower()), None)
if age_col:
    for c in num_cols:
        if c != age_col:
            df[f'{c}_per_age'] = df[c] / (df[age_col] + 1)
            break  # just one example ratio to keep things simple

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 29 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   patient_id                 9000 non-null   int64  
 1   age                        9000 non-null   int64  
 2   sex                        9000 non-null   int64  
 3   resting_bp_systolic        9000 non-null   int64  
 4   resting_bp_diastolic       9000 non-null   int64  
 5   cholesterol_total          9000 non-null   int64  
 6   hdl                        9000 non-null   int64  
 7   ldl                        9000 non-null   int64  
 8   triglycerides              9000 non-null   int64  
 9   fasting_blood_sugar        9000 non-null   int64  
 10  hba1c                      9000 non-null   float64
 11  bmi                        9000 non-null   float64
 12  resting_heart_rate         9000 non-null   int64  
 13  max_heart_rate_achieved    9000 non-null   int64

# Model Training

In [7]:
X = df.drop(target_col, axis=1)
y = df[target_col]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train, y_train)

model_dt = DecisionTreeClassifier(random_state=42)
model_dt.fit(X_train, y_train)

model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(X_train, y_train)

print('Models trained: Logistic Regression, Decision Tree, Random Forest')

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Models trained: Logistic Regression, Decision Tree, Random Forest


# Model Evaluation

In [8]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    'Logistic Regression': model_lr,
    'Decision Tree': model_dt,
    'Random Forest': model_rf
}

results = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f'--- {name} ---')
    print('Accuracy:', acc)
    print(classification_report(y_test, y_pred))
    print('Confusion Matrix:')
    print(confusion_matrix(y_test, y_pred))
    print()

best_model_name = max(results, key=results.get)
print('Best model:', best_model_name, 'with accuracy', results[best_model_name])

--- Logistic Regression ---
Accuracy: 0.8638888888888889
              precision    recall  f1-score   support

           0       0.88      0.93      0.90      1255
           1       0.81      0.72      0.76       545

    accuracy                           0.86      1800
   macro avg       0.85      0.82      0.83      1800
weighted avg       0.86      0.86      0.86      1800

Confusion Matrix:
[[1163   92]
 [ 153  392]]

--- Decision Tree ---
Accuracy: 0.8144444444444444
              precision    recall  f1-score   support

           0       0.87      0.87      0.87      1255
           1       0.69      0.69      0.69       545

    accuracy                           0.81      1800
   macro avg       0.78      0.78      0.78      1800
weighted avg       0.81      0.81      0.81      1800

Confusion Matrix:
[[1088  167]
 [ 167  378]]

--- Random Forest ---
Accuracy: 0.8788888888888889
              precision    recall  f1-score   support

           0       0.89      0.95      0

# Hyperparameter tuning

In [9]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=kf,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print('Best params:', grid_search.best_params_)
print('Best CV accuracy:', grid_search.best_score_)

final_model = grid_search.best_estimator_
y_pred_final = final_model.predict(X_test)
print('Test accuracy after tuning:', accuracy_score(y_test, y_pred_final))
print(classification_report(y_test, y_pred_final))

Best params: {'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 100}
Best CV accuracy: 0.8834722222222222
Test accuracy after tuning: 0.8833333333333333
              precision    recall  f1-score   support

           0       0.89      0.95      0.92      1255
           1       0.86      0.74      0.79       545

    accuracy                           0.88      1800
   macro avg       0.87      0.84      0.86      1800
weighted avg       0.88      0.88      0.88      1800



# Model dump

In [10]:
import pickle
import joblib

with open('heart_disease_risk_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

joblib.dump(final_model, 'heart_disease_risk_model.joblib')

print('Model saved as heart_disease_risk_model.pkl and heart_disease_risk_model.joblib')

Model saved as heart_disease_risk_model.pkl and heart_disease_risk_model.joblib
